In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import sys
sys.path.append("..")  # so imports from src/ work

# Load raw 3200 Hz data
file_path = "../data/raw/motor_raw_3200Hz.xlsx"   # adjust path if needed
df = pd.read_excel(file_path)

df.head()


,sample_index,time_sec,acc_x,acc_y,acc_z,mic,label
0,0,0.000000,0.009934,0.014640,-0.003786,30.515058,0
1,1,0.000313,0.026640,0.000103,0.031240,30.137839,0
2,2,0.000625,0.071481,0.050766,0.064738,32.235947,0
3,3,0.000937,0.117546,0.068141,0.119688,30.099500,0
4,4,0.001250,0.110122,0.129384,0.153240,32.229082,0


In [2]:
df.info()
df['label'].value_counts()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64000 entries, 0 to 63999
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sample_index  64000 non-null  int64  
 1   time_sec      64000 non-null  float64
 2   acc_x         64000 non-null  float64
 3   acc_y         64000 non-null  float64
 4   acc_z         64000 non-null  float64
 5   mic           64000 non-null  float64
 6   label         64000 non-null  int64  
dtypes: float64(5), int64(2)
memory usage: 3.4 MB


label
0    32000
1    32000
Name: count, dtype: int64

In [3]:
fs = 3200
WIN_SEC = 1
WIN_SIZE = int(fs * WIN_SEC)

n_samples = len(df)
total_windows = n_samples // WIN_SIZE

print("Total Samples:", n_samples)
print("Window Size:", WIN_SIZE)
print("Total Windows:", total_windows)


Total Samples: 64000
Window Size: 3200
Total Windows: 20


In [4]:
fs = 3200
WIN_SEC = 1
WIN_SIZE = int(fs * WIN_SEC)

n_samples = len(df)
total_windows = n_samples // WIN_SIZE

print("Total Samples:", n_samples)
print("Window Size:", WIN_SIZE)
print("Total Windows:", total_windows)


Total Samples: 64000
Window Size: 3200
Total Windows: 20


In [5]:
from scipy.stats import skew, kurtosis
from scipy.fft import rfft, rfftfreq

def extract_features(window, fs):
    feats = {}
    channels = ['acc_x', 'acc_y', 'acc_z', 'mic']
    
    for ch in channels:
        sig = window[ch].values
        prefix = f"{ch}_"

        feats[prefix+"mean"] = sig.mean()
        feats[prefix+"std"] = sig.std()
        feats[prefix+"rms"] = np.sqrt(np.mean(sig**2))
        feats[prefix+"min"] = sig.min()
        feats[prefix+"max"] = sig.max()
        feats[prefix+"skew"] = skew(sig)
        feats[prefix+"kurt"] = kurtosis(sig)
        feats[prefix+"ptp"] = np.ptp(sig)  # FIX for NumPy 2.0
        feats[prefix+"crest"] = feats[prefix+"max"] / (feats[prefix+"rms"] + 1e-8)

        # Frequency domain
        fft_vals = np.abs(rfft(sig))
        freqs = rfftfreq(len(sig), 1/fs)

        feats[prefix+"dom_freq"] = freqs[np.argmax(fft_vals)]
        feats[prefix+"spec_centroid"] = float(np.sum(freqs * fft_vals) / (np.sum(fft_vals) + 1e-8))

    return feats


In [6]:
feature_rows = []

for w in range(total_windows):
    start = w * WIN_SIZE
    end   = start + WIN_SIZE
    window = df.iloc[start:end]

    feats = extract_features(window, fs)
    feats["label"] = int(window["label"].mode()[0])
    feats["window_id"] = w
    
    feature_rows.append(feats)

feat_df = pd.DataFrame(feature_rows)

feat_df.head()


,acc_x_mean,acc_x_std,acc_x_rms,acc_x_min,acc_x_max,acc_x_skew,acc_x_kurt,acc_x_ptp,acc_x_crest,acc_x_dom_freq,...,mic_min,mic_max,mic_skew,mic_kurt,mic_ptp,mic_crest,mic_dom_freq,mic_spec_centroid,label,window_id
0,0.000426,0.212640,0.212640,-0.345890,0.377080,0.005982,-1.477080,0.722970,1.773324,50.0,...,25.101960,35.301495,0.030041,-0.704463,10.199535,1.175696,0.0,358.986322,0,0
1,-0.000496,0.212703,0.212703,-0.355648,0.357277,0.004786,-1.469433,0.712925,1.679694,50.0,...,24.428052,35.253602,-0.017947,-0.623656,10.825549,1.173343,0.0,352.017830,0,1
2,0.000001,0.213153,0.213153,-0.351236,0.354400,-0.001945,-1.470361,0.705636,1.662651,50.0,...,25.066049,35.013281,-0.027233,-0.652069,9.947232,1.165571,0.0,358.356840,0,2
3,-0.000224,0.213240,0.213240,-0.363207,0.342706,0.002403,-1.476241,0.705913,1.607142,50.0,...,25.060667,35.160818,0.021687,-0.668280,10.100150,1.168555,0.0,355.663401,0,3
4,0.000444,0.213498,0.213498,-0.351114,0.348042,-0.001477,-1.473473,0.699156,1.630186,50.0,...,24.881804,35.403695,0.008718,-0.647834,10.521891,1.177954,0.0,357.224918,0,4


In [7]:
output_path = "../data/processed/smds_features.csv"

# Create folder if doesn't exist
Path("../data/processed").mkdir(parents=True, exist_ok=True)

feat_df.to_csv(output_path, index=False)
output_path


'../data/processed/smds_features.csv'

In [8]:
print("Feature dataset created!")
print("Shape:", feat_df.shape)
print("Healthy windows:", (feat_df.label==0).sum())
print("Faulty windows:", (feat_df.label==1).sum())


Feature dataset created!
Shape: (20, 46)
Healthy windows: 10
Faulty windows: 10
